# Multi-Domain Threat Intelligence & Attribution Engine

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
import networkx as nx
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import LatentDirichletAllocation
from scipy.stats import entropy, chi2_contingency
from scipy.spatial.distance import cosine, jaccard
import hashlib
from collections import defaultdict, Counter
import re

class ThreatIntelligenceAttributionEngine:
    def __init__(self, confidence_threshold=0.7):
        self.apt_classifier = GradientBoostingClassifier(n_estimators=200, random_state=42)
        self.campaign_detector = RandomForestClassifier(n_estimators=150, random_state=42)
        self.clustering_model = AgglomerativeClustering(n_clusters=None, distance_threshold=0.6)
        self.malware_vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1,4))
        self.infrastructure_vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1,2))
        self.topic_model = LatentDirichletAllocation(n_components=20, random_state=42)
        self.scaler = StandardScaler()
        self.apt_signatures = defaultdict(dict)
        self.campaign_graph = nx.DiGraph()
        self.confidence_threshold = confidence_threshold

    def extract_malware_features(self, malware_samples):
        features = []
        textual_features = []

        for sample in malware_samples:
            binary_entropy = self.calculate_entropy(sample.get('binary_data', b''))
            file_size = len(sample.get('binary_data', b''))

            strings = sample.get('strings', [])
            api_calls = sample.get('api_calls', [])
            network_indicators = sample.get('network_iocs', [])

            string_entropy = np.mean([self.calculate_entropy(s.encode()) for s in strings]) if strings else 0
            api_diversity = len(set(api_calls)) / max(len(api_calls), 1)
            network_entropy = entropy([len(ioc) for ioc in network_indicators] + [1]) if network_indicators else 0

            packer_indicators = sum(1 for s in strings if any(p in s.lower() for p in ['upx', 'aspack', 'pecompact', 'vmprotect']))
            crypto_indicators = sum(1 for api in api_calls if any(c in api.lower() for c in ['crypt', 'hash', 'encrypt', 'decrypt']))
            persistence_indicators = sum(1 for api in api_calls if any(p in api.lower() for p in ['registry', 'service', 'startup', 'schedule']))

            code_reuse_score = self.calculate_code_similarity(sample.get('code_blocks', []))
            obfuscation_score = self.detect_obfuscation_techniques(strings, api_calls)

            features.append([binary_entropy, file_size, string_entropy, api_diversity, network_entropy,
                           packer_indicators, crypto_indicators, persistence_indicators, code_reuse_score, obfuscation_score])

            textual_content = ' '.join(strings + api_calls + [str(ioc) for ioc in network_indicators])
            textual_features.append(textual_content)

        return np.array(features), textual_features

    def calculate_entropy(self, data):
        if isinstance(data, str):
            data = data.encode()
        if len(data) == 0:
            return 0
        byte_counts = Counter(data)
        probs = np.array(list(byte_counts.values())) / len(data)
        return -np.sum(probs * np.log2(probs + 1e-10))

    def calculate_code_similarity(self, code_blocks):
        if len(code_blocks) < 2:
            return 0
        similarities = []
        for i in range(len(code_blocks)):
            for j in range(i+1, min(i+10, len(code_blocks))):
                block1_hash = hashlib.md5(str(code_blocks[i]).encode()).hexdigest()
                block2_hash = hashlib.md5(str(code_blocks[j]).encode()).hexdigest()
                similarity = 1 - (bin(int(block1_hash, 16) ^ int(block2_hash, 16)).count('1') / 128)
                similarities.append(similarity)
        return np.mean(similarities) if similarities else 0

    def detect_obfuscation_techniques(self, strings, api_calls):
        obfuscation_score = 0

        for string in strings:
            if len(re.findall(r'[A-Za-z0-9+/]{20,}', string)) > 0:
                obfuscation_score += 0.1
            if string.count('\\x') > 5:
                obfuscation_score += 0.1

        dynamic_api_loading = sum(1 for api in api_calls if 'getprocaddress' in api.lower() or 'loadlibrary' in api.lower())
        obfuscation_score += min(dynamic_api_loading * 0.05, 0.3)

        return min(obfuscation_score, 1.0)

    def analyze_infrastructure_patterns(self, infrastructure_data):
        features = []
        domain_features = []

        for infra in infrastructure_data:
            domains = infra.get('domains', [])
            ips = infra.get('ip_addresses', [])
            certificates = infra.get('ssl_certificates', [])

            domain_entropy = np.mean([self.calculate_entropy(d) for d in domains]) if domains else 0
            ip_geolocation_diversity = len(set(infra.get('geolocations', [])))
            cert_validity_variance = np.var([c.get('validity_days', 365) for c in certificates]) if certificates else 0

            dga_indicators = sum(1 for d in domains if self.detect_dga_domain(d))
            bulletproof_hosting_score = sum(1 for geo in infra.get('geolocations', []) if geo in ['RU', 'CN', 'KP'])

            fast_flux_indicators = len(set(ips)) / max(len(domains), 1) if domains else 0
            dns_ttl_variance = np.var([infra.get('dns_ttl', 300) for _ in domains]) if domains else 0

            features.append([domain_entropy, ip_geolocation_diversity, cert_validity_variance,
                           dga_indicators, bulletproof_hosting_score, fast_flux_indicators, dns_ttl_variance])

            infra_text = ' '.join(domains + [str(ip) for ip in ips] + [c.get('issuer', '') for c in certificates])
            domain_features.append(infra_text)

        return np.array(features), domain_features

    def detect_dga_domain(self, domain):
        vowels = 'aeiou'
        consonants = 'bcdfghjklmnpqrstvwxyz'

        if len(domain) < 6:
            return False

        vowel_ratio = sum(1 for c in domain if c in vowels) / len(domain)
        consonant_clusters = len(re.findall(r'[bcdfghjklmnpqrstvwxyz]{3,}', domain))
        digit_ratio = sum(1 for c in domain if c.isdigit()) / len(domain)

        entropy_score = self.calculate_entropy(domain.encode())

        dga_score = (abs(vowel_ratio - 0.4) * 2 + consonant_clusters * 0.3 +
                    digit_ratio * 0.5 + entropy_score / 8)

        return dga_score > 1.2

    def perform_campaign_clustering(self, malware_features, infrastructure_features, temporal_data):
        combined_features = []

        for i, (mal_feat, infra_feat) in enumerate(zip(malware_features, infrastructure_features)):
            temporal_feat = temporal_data[i] if i < len(temporal_data) else {}

            time_clustering = self.analyze_temporal_patterns(temporal_feat.get('timestamps', []))
            target_diversity = len(set(temporal_feat.get('target_sectors', [])))
            geographic_spread = len(set(temporal_feat.get('target_countries', [])))

            combined_feat = np.concatenate([mal_feat, infra_feat, [time_clustering, target_diversity, geographic_spread]])
            combined_features.append(combined_feat)

        if len(combined_features) > 1:
            features_scaled = self.scaler.fit_transform(combined_features)
            cluster_labels = self.clustering_model.fit_predict(features_scaled)
            return cluster_labels
        return [0] * len(combined_features)

    def analyze_temporal_patterns(self, timestamps):
        if len(timestamps) < 3:
            return 0

        timestamps = np.sort(timestamps)
        intervals = np.diff(timestamps)

        periodicity_score = 0
        for period in [86400, 604800, 2592000]:
            matches = sum(1 for interval in intervals if abs(interval - period) < period * 0.1)
            periodicity_score += matches / len(intervals)

        return min(periodicity_score, 1.0)

    def attribute_to_threat_actor(self, sample_features, known_apt_profiles):
        attribution_scores = {}

        for apt_name, profile in known_apt_profiles.items():
            similarity_scores = []

            malware_similarity = cosine(sample_features['malware'], profile.get('malware_signature', [0]*10))
            infrastructure_similarity = cosine(sample_features['infrastructure'], profile.get('infrastructure_signature', [0]*7))

            ttps_overlap = jaccard(set(sample_features.get('ttps', [])), set(profile.get('ttps', [])))
            target_overlap = jaccard(set(sample_features.get('targets', [])), set(profile.get('targets', [])))

            temporal_similarity = abs(sample_features.get('campaign_duration', 30) - profile.get('avg_campaign_duration', 30)) / 365

            final_score = (1 - malware_similarity) * 0.3 + (1 - infrastructure_similarity) * 0.25 + ttps_overlap * 0.2 + target_overlap * 0.15 + (1 - temporal_similarity) * 0.1
            attribution_scores[apt_name] = final_score

        return sorted(attribution_scores.items(), key=lambda x: x[1], reverse=True)

    def generate_threat_intelligence_report(self, malware_samples, infrastructure_data, temporal_data, known_apts):
        malware_features, malware_text = self.extract_malware_features(malware_samples)
        infrastructure_features, infra_text = self.analyze_infrastructure_patterns(infrastructure_data)

        campaign_clusters = self.perform_campaign_clustering(malware_features, infrastructure_features, temporal_data)

        malware_topics = self.topic_model.fit_transform(self.malware_vectorizer.fit_transform(malware_text))

        threat_intelligence = {
            'campaigns_identified': len(set(campaign_clusters)),
            'malware_families': [],
            'infrastructure_clusters': [],
            'attribution_results': [],
            'threat_landscape_topics': [],
            'confidence_metrics': {}
        }

        for cluster_id in set(campaign_clusters):
            cluster_indices = [i for i, c in enumerate(campaign_clusters) if c == cluster_id]

            if cluster_indices:
                cluster_malware_features = malware_features[cluster_indices[0]] if len(cluster_indices) > 0 else np.zeros(10)
                cluster_infra_features = infrastructure_features[cluster_indices[0]] if len(cluster_indices) > 0 else np.zeros(7)

                sample_features = {
                    'malware': cluster_malware_features,
                    'infrastructure': cluster_infra_features,
                    'ttps': [f'ttp_{i}' for i in range(5)],
                    'targets': [f'sector_{i}' for i in range(3)],
                    'campaign_duration': 45
                }

                attributions = self.attribute_to_threat_actor(sample_features, known_apts)

                threat_intelligence['attribution_results'].append({
                    'cluster_id': cluster_id,
                    'sample_count': len(cluster_indices),
                    'top_attribution': attributions[0] if attributions else ('Unknown', 0.0),
                    'confidence': attributions[0][1] if attributions else 0.0
                })

        for topic_idx, topic in enumerate(malware_topics.T):
            if np.sum(topic) > 0.1:
                threat_intelligence['threat_landscape_topics'].append({
                    'topic_id': topic_idx,
                    'prevalence': np.sum(topic),
                    'samples_affected': np.sum(topic > 0.1)
                })

        avg_confidence = np.mean([attr['confidence'] for attr in threat_intelligence['attribution_results']]) if threat_intelligence['attribution_results'] else 0
        threat_intelligence['confidence_metrics'] = {
            'overall_confidence': avg_confidence,
            'high_confidence_attributions': sum(1 for attr in threat_intelligence['attribution_results'] if attr['confidence'] > self.confidence_threshold)
        }

        return threat_intelligence